## 化学信息学工具模块
  - **1. SMILES 规范化**   `canonical_smiles()`
  - **2. InChIKey14 计算**   `get_inchikey14()`
  - **3. 分子式提取**   `get_molecular_formula()`
  - **4. 分子同一性判断**   `get_molecular_formula()`
  - **5. 候选去重**   `is_same_molecule()`
  - **6. 排名评估**   `evaluate_rank()`

In [2]:
import sys
print("Python 路径:", sys.executable)

from rdkit import rdBase
print("RDKit 版本:", rdBase.rdkitVersion)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", plt.matplotlib.__version__)

Python 路径: /public/home/zhaoshuo/miniconda3/envs/casmi_env/bin/python
RDKit 版本: 2026.03.6
numpy: 2.2.6
pandas: 2.3.3
matplotlib: 3.10.9


In [3]:
from rdkit import Chem
from rdkit.Chem import inchi, rdMolDescriptors
from rdkit import RDLogger

In [1]:
# smiles standardization
def canonical_smiles(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)

In [4]:
canonical_smiles("OC1=CC=CC=C1")

'Oc1ccccc1'

In [12]:
# smiles to InChIKey14
def get_inchikey14(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    key = inchi.MolToInchiKey(mol)
    if not key:
        return None
    return key.split("-")[0]

In [6]:
get_inchikey14("Oc1ccccc1")

'ISWSIDIOOBJBQZ'

In [7]:
# smiles to formula
def get_molecular_formula(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return rdMolDescriptors.CalcMolFormula(mol)

In [11]:
get_molecular_formula("OC1=CC=CC=C1")
get_molecular_formula("Oc1ccccc1")

'C6H6O'

In [1]:
# 判断是否为同一分子
def is_same_molecule(smiles_a: str, smiles_b: str) -> bool:
    ka = get_inchikey14(smiles_a)
    kb = get_inchikey14(smiles_b)
    return ka is not None and ka == kb

In [10]:
is_same_molecule("c1ccccc1", "C1=CC=CC=C1")

True

In [14]:
# 候选分子去重
def deduplicate_candidates(candidates: list[str]) -> list[str]:
    seen = set()
    unique = []
    for smi in candidates:
        key = get_inchikey14(smi)
        if key is None:
            continue
        if key in seen:
            continue
        seen.add(key)
        unique.append(smi)
    return unique #输出为 smile 格式

In [15]:
candidates = ["c1ccccc1", "C1=CC=CC=C1", "Cc1ccccc1", "c1ccccc1O"]
deduplicate_candidates(candidates)

['c1ccccc1', 'Cc1ccccc1', 'c1ccccc1O']

In [19]:
# 排名计算
def evaluate_rank(predicted_list: list[str], answer_smiles: str) -> int | None:
    target = get_inchikey14(answer_smiles)
    if target is None:
        return None
    for rank, smi in enumerate(predicted_list, start=1):
        if get_inchikey14(smi) == target:
            return rank
    return None

In [18]:
preds = ["Cc1ccccc1", "c1ccccc1O", "c1ccccc1"]
answer = "c1ccccc1"
evaluate_rank(preds, answer)

3